# Llama 3.1 8B QLoRA Fine-Tuning with Unsloth
This notebook fine-tunes Llama 3.1 8B using PEFT/QLoRA on the 1000 balanced dataset, evaluates it, and saves the model.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Load and Prepare the Balanced Dataset
Upload `erp_whw_1000_balanced.jsonl` to Colab before running this.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")

# Load the 1000 balanced dataset
dataset = load_dataset("json", data_files="erp_whw_1000_balanced.jsonl", split="train")

# Split into train and eval (e.g., 90% train, 10% eval)
dataset = dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

def format_prompts(examples):
    conversations = []
    for instruction, input_str, output in zip(examples["instruction"], examples["input"], examples["output"]):
        system_prompt = "You are an expert ERP Finance AI assistant. You process financial events accurately and do not hallucinate outside the provided context."
        user_content = instruction
        if input_str and input_str.strip():
            user_content += f"\nContext: {input_str}"
        convo = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output}
        ]
        conversations.append(convo)
    formatted_texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in conversations]
    return { "text" : formatted_texts }

train_dataset = train_dataset.map(format_prompts, batched = True)
eval_dataset = eval_dataset.map(format_prompts, batched = True)

print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")

### Train with Evaluation

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Packing disabled to keep evaluation metrics clean
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1, # Adjust epochs as needed
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### Evaluation Methods
Let's run an evaluation manually to see the loss, and then test the model with a sample prompt.

In [ ]:
# 1. Evaluate the model quantitatively
# Extracting eval_loss from training logs to avoid 'on_train_begin' bug in Colab.
import math
eval_logs = [log for log in trainer.state.log_history if 'eval_loss' in log]
if eval_logs:
    last_eval = eval_logs[-1]
    print(f"Final Evaluation Results: {last_eval}")
    print(f"Perplexity: {math.exp(last_eval['eval_loss']):.2f}")
else:
    print("No evaluation logs found.")

In [ ]:
# 2. Qualitative Evaluation (Inference)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "system", "content": "You are an expert ERP Finance AI assistant. You process financial events accurately and do not hallucinate outside the provided context."},
    {"role": "user", "content": "Classify the following into an ERP transaction.\nContext: Bought office supplies for $500."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

### Save the Model

In [ ]:
# Save LoRA weights locally
model.save_pretrained("erp_finance_lora_1000") 
tokenizer.save_pretrained("erp_finance_lora_1000")

# Export to GGUF format for Ollama
# 'q4_k_m' is a standard 4-bit quantization that balances size and performance
model.save_pretrained_gguf(
    "erp_finance_gguf_1000", 
    tokenizer, 
    quantization_method = "q4_k_m"
)
print("Saved successfully. Download the GGUF files from the erp_finance_gguf_1000 folder.")

In [ ]:
from google.colab import files
import os

# The folder we saved the GGUF to is 'erp_finance_gguf_1000'
output_dir = "erp_finance_gguf_1000"

if os.path.exists(output_dir):
    gguf_files = [f for f in os.listdir(output_dir) if f.endswith(".gguf")]
    if gguf_files:
        gguf_file = os.path.join(output_dir, gguf_files[0])
        print(f"Downloading: {gguf_file}")
        files.download(gguf_file)
    else:
        print("No GGUF file found in the directory.")
else:
    print(f"Directory {output_dir} does not exist.")